In [ ]:
import requests
import torch
from PIL import Image
from io import BytesIO

from transformers import AutoProcessor, AutoModelForVision2Seq
from transformers.image_utils import load_image

In [12]:
import pandas as pd
import os
from tqdm import tqdm
import torch

model_name = 'idefics'
prompt_type = 'zeroshot'
dataset = 'type1'
image_type = 'combined'
device = "cuda:1"

qas = pd.read_json(f'../{dataset}/qa.json')
image_indices = qas['image_index'].values.astype(int)
questions = qas['question'].values
answers = qas['answer'].values
image_base_path = f'../{dataset}/{image_type}/'

all_images = os.listdir(image_base_path)
index_to_image = {}

if dataset == 'type1':
    prefix = 'multichart_'
    if image_type == 'original' or image_type == 'simple':
        sep = '_'
    else:
        sep = '.'

for index in set(image_indices):
    for image in all_images:
        if image.startswith(f'{prefix}{index}{sep}'):
            if index not in index_to_image:
                index_to_image[index] = []
            index_to_image[index].append(image)

message_template = [
    {
        "role": "user",
        "content": []
    }
]

def get_message(image_index, prompt, question):
    content = []
    images = []
    for image in index_to_image[image_index]:
        image_path = f'{image_base_path}{image}'
        image = Image.open(image_path).convert('RGB')
        images.append(image)
        content.append({"type": "image"})
    message = message_template.copy()
    content.append({"type" : "text", "text": prompt.format(question)})
    message[0]['content'] = content
    return message, images

In [ ]:
get_message(5, "hi", "hihi")

In [ ]:
DEVICE = "cuda:0"
processor = AutoProcessor.from_pretrained("HuggingFaceM4/Idefics3-8B-Llama3")
model = AutoModelForVision2Seq.from_pretrained(
    "HuggingFaceM4/Idefics3-8B-Llama3", torch_dtype=torch.bfloat16, cache_dir = '/media/vivek/drive/multichartqa/models_cache'
).to(DEVICE)

In [ ]:
message, images = get_message(5, "Describe this image", "")

In [21]:
generated_ids = model.generate(**inputs, max_new_tokens=1024)
generated_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)

In [ ]:
model = model.eval()

model_responses = []

def run_model():
    for i, question in enumerate(tqdm(questions)):
        image_index = image_indices[i]
        prompt = """Your task is the answer the question based on the given image. Your final answer to the question should strictly be in the format - "Final Answer:" <final_answer>.\n\nQuestion: {}""" 
        message, images = get_message(image_index, prompt, question)
        prompt = processor.apply_chat_template(message, add_generation_prompt=True)
        inputs = processor(text=prompt, images=images, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        generated_ids = model.generate(**inputs, max_new_tokens=1024)
        answer = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        model_responses.append({'question_id': i, 'question': question, 'gold': answers[i], 'response': answer.strip()})
        # print(f'Question: {question}\nAnswer: {answers[i]}\nResponse: {answer.strip()}\n\n')
        
if __name__ == '__main__':
    run_model()
    os.makedirs(f'../model_responses/{dataset}', exist_ok=True)

    model_responses_df = pd.DataFrame(model_responses)
    model_responses_df.to_json(f'../model_responses/{dataset}/{model_name}_{image_type}_{prompt_type}.json', orient='records')